## Apache Spark Versus Pandas in Databricks Environment

In [ ]:
import time
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

TOTAL_LINHAS = 200_000
print(f"Iniciando simulação com {TOTAL_LINHAS:,} cenários contábeis...")

# -------------------------------------------------------------
# 1. Geração Sintética Distribuída no Spark (200k cenários)
# -------------------------------------------------------------
# Gera o DataFrame Template (Expectativa)
df_template_200k = spark.range(1, TOTAL_LINHAS + 1).select(
    F.col("id").cast("int").alias("id_cenario"),
    F.concat(F.lit("ROT_"), F.lpad(F.col("id"), 7, "0")).alias("chave_roteiro"),
    F.lit("1.1.1.01").alias("conta_debito_esperada"),
    F.lit("2.1.1.01").alias("conta_credito_esperada"),
    F.lit(1500.0).alias("valor_esperado")
)

# Gera o DataFrame Snapshot (Simulando 10% de órfãos e ~4,6% de divergentes)
# 180.000 registros gerados (20.000 órfãos)
df_snapshot_200k = spark.range(1, int(TOTAL_LINHAS * 0.90) + 1).select(
    F.col("id").cast("int").alias("id_cenario"),
    F.concat(F.lit("ROT_"), F.lpad(F.col("id"), 7, "0")).alias("chave_roteiro"),
    # Injeta conta transitória 1.1.9.99 nos primeiros 9.200 registros (4,6%)
    F.when(F.col("id") <= int(TOTAL_LINHAS * 0.046), F.lit("1.1.9.99"))
     .otherwise(F.lit("1.1.1.01")).alias("conta_debito_gerada"),
    F.lit("2.1.1.01").alias("conta_credito_gerada"),
    F.lit(1500.0).alias("valor_gerado")
)

# -------------------------------------------------------------
# 2. Execução e Benchmark: Apache Spark 4.2.0 (Databricks)
# -------------------------------------------------------------
inicio_spark = time.perf_counter()

df_join_200k = df_template_200k.alias("t").join(
    df_snapshot_200k.alias("s"),
    on="id_cenario",
    how="left"
)

df_conciliado_200k = df_join_200k.withColumn(
    "status_homologacao",
    F.when(F.col("s.id_cenario").isNull(), "Não Sensibilizado")
     .when(
         (F.col("t.conta_debito_esperada") != F.col("s.conta_debito_gerada")) | 
         (F.col("t.conta_credito_esperada") != F.col("s.conta_credito_gerada")) |
         (F.col("t.valor_esperado") != F.col("s.valor_gerado")), 
         "Divergente"
     )
     .otherwise("Sensibilizado com Sucesso")
)

resumo_spark_200k = df_conciliado_200k.groupBy("status_homologacao").count()
contagem_spark = {row["status_homologacao"]: row["count"] for row in resumo_spark_200k.collect()}
tempo_spark_200k = time.perf_counter() - inicio_spark

print(f"\n[Apache Spark 4.2.0]")
print(f"Tempo de execução: {tempo_spark_200k:.4f} segundos")
print(f"Resultados: {contagem_spark}")

# -------------------------------------------------------------
# 3. Execução e Benchmark: Python Pandas (Em Memória Local)
# -------------------------------------------------------------
inicio_pandas = time.perf_counter()

# Converte para Pandas simulando o payload no backend local
pdf_template_200k = df_template_200k.toPandas()
pdf_snapshot_200k = df_snapshot_200k.toPandas()

pdf_join_200k = pd.merge(pdf_template_200k, pdf_snapshot_200k, on="id_cenario", how="left")

condicoes = [
    pdf_join_200k["chave_roteiro_y"].isna(),
    (pdf_join_200k["conta_debito_esperada"] != pdf_join_200k["conta_debito_gerada"]) |
    (pdf_join_200k["conta_credito_esperada"] != pdf_join_200k["conta_credito_gerada"]) |
    (pdf_join_200k["valor_esperado"] != pdf_join_200k["valor_gerado"])
]
escolhas = ["Não Sensibilizado", "Divergente"]
pdf_join_200k["status_homologacao"] = np.select(condicoes, escolhas, default="Sensibilizado com Sucesso")

contagem_pandas = pdf_join_200k["status_homologacao"].value_counts().to_dict()
tempo_pandas_200k = time.perf_counter() - inicio_pandas

print(f"\n[Pandas / Microsserviço Local]")
print(f"Tempo de execução: {tempo_pandas_200k:.4f} segundos")
print(f"Resultados: {contagem_pandas}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# 1. Configuração da Figura
fig, ax = plt.subplots(figsize=(14, 10), dpi=300)
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis("off")

# Paleta de Cores Acadêmica / Corporativa
c_blue = "#1A73E8"
c_bg_blue = "#E8F0FE"
c_amber = "#EA8600"
c_bg_amber = "#FEF7E0"
c_green = "#137333"
c_bg_green = "#E6F4EA"
c_gray = "#5F6368"
c_border = "#3C4043"

# Função utilitária para desenhar caixas de processo
def draw_box(x, y, w, h, title, text, bg_color, border_color):
    box = patches.FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.2,rounding_size=0.15",
        facecolor=bg_color,
        edgecolor=border_color,
        linewidth=1.5
    )
    ax.add_patch(box)
    ax.text(x + w/2, y + h - 0.35, title, ha="center", va="center", fontsize=11, fontweight="bold", color=border_color)
    ax.text(x + w/2, y + (h - 0.4)/2, text, ha="center", va="center", fontsize=9, color="#202124", multialignment="center")

# -------------------------------------------------------------
# CAMADA 1: Processamento Distribuído
# -------------------------------------------------------------
sub1 = patches.Rectangle((0.5, 5.2), 13, 4.3, fill=False, edgecolor="#BDC1C6", linestyle="--", linewidth=1.5)
ax.add_patch(sub1)
ax.text(0.8, 9.2, "1. Processamento Distribuído (Databricks Serverless / Spark 4.2.0)", fontsize=11, fontweight="bold", color=c_blue)

# Caixas de entrada e Spark
draw_box(1.0, 7.2, 3.2, 1.4, "Fontes Descaracterizadas", "• TemplateExpectativa\n• SnapshotProcessado\n(Tipagem StructType)", c_bg_blue, c_blue)
draw_box(5.4, 7.2, 3.2, 1.4, "Conciliação Distribuída", "Left Join sobre id_cenario\nApache Spark 4.2.0\n(0,501s / 200k em 0,684s)", c_bg_blue, c_blue)
draw_box(9.8, 7.2, 3.2, 1.4, "Métricas & Amostras", "• 85,4% Sucesso (427 / 170.8k)\n• 10,0% Órfãos (50 / 20k)\n• 4,6% Divergentes (23 / 9.2k)", c_bg_blue, c_blue)
draw_box(5.4, 5.5, 7.6, 1.2, "Serialização e Desacoplamento de Payload", "Agregação volumétrica + Amostra de anomalias paramétricas\n(Garante tempo de inferência invariante na IA)", "#FFFFFF", c_gray)

# Setas da Camada 1
ax.annotate("", xy=(5.4, 7.9), xytext=(4.2, 7.9), arrowprops=dict(arrowstyle="-|>", lw=2, color=c_blue))
ax.annotate("", xy=(9.8, 7.9), xytext=(8.6, 7.9), arrowprops=dict(arrowstyle="-|>", lw=2, color=c_blue))
ax.annotate("", xy=(9.2, 6.7), xytext=(9.8, 7.2), arrowprops=dict(arrowstyle="-|>", lw=1.5, color=c_gray))

# -------------------------------------------------------------
# CAMADA 2: Diagnóstico Cognitivo
# -------------------------------------------------------------
sub2 = patches.Rectangle((0.5, 2.7), 6.2, 2.2, fill=False, edgecolor="#BDC1C6", linestyle="--", linewidth=1.5)
ax.add_patch(sub2)
ax.text(0.8, 4.6, "2. Auditoria Cognitiva (Google Gemini 3.6 Flash)", fontsize=11, fontweight="bold", color=c_amber)

draw_box(1.0, 2.9, 5.2, 1.4, "Agente Cognitivo & Pydantic", "Modelo: gemini-3.6-flash (temp=0.1)\n• Diagnóstico de Causa-Raiz (Conta 1.1.9.99)\n• Parecer SOX 404: 'Em Risco Material'\n• Recomendações Prescritivas", c_bg_amber, c_amber)

# Seta do Spark para o Gemini
ax.annotate("", xy=(3.6, 4.3), xytext=(5.4, 6.1), arrowprops=dict(arrowstyle="-|>", lw=2, color=c_amber, connectionstyle="arc3,rad=0.2"))
ax.text(3.5, 5.4, "Payload JSON (Resumo)", fontsize=8.5, color=c_amber, fontweight="bold")

# -------------------------------------------------------------
# CAMADA 3: Persistência e Governança
# -------------------------------------------------------------
sub3 = patches.Rectangle((7.3, 2.7), 6.2, 2.2, fill=False, edgecolor="#BDC1C6", linestyle="--", linewidth=1.5)
ax.add_patch(sub3)
ax.text(7.6, 4.6, "3. Persistência e Governança (Delta Lake)", fontsize=11, fontweight="bold", color=c_green)

draw_box(7.8, 2.9, 5.2, 1.4, "Unity Catalog / Tabelas Delta", "• auditoria_performance_execucoes\n  (Trilha de Auditoria SOX / Change Data Feed)\n• auditoria_performance_estatisticas\n  (Estatísticas de Latência / Time Travel)", c_bg_green, c_green)

# Setas para Persistência
ax.annotate("", xy=(10.4, 4.3), xytext=(10.4, 5.5), arrowprops=dict(arrowstyle="-|>", lw=1.5, color=c_green, linestyle=":"))
ax.annotate("", xy=(7.8, 3.6), xytext=(6.2, 3.6), arrowprops=dict(arrowstyle="-|>", lw=2, color=c_green))
ax.text(6.4, 3.8, "Parecer Final", fontsize=8.5, color=c_green, fontweight="bold")

# -------------------------------------------------------------
# Rodapé e Salvamento da Imagem
# -------------------------------------------------------------
ax.text(7.0, 0.6, "Figura 1 – Diagrama da Arquitetura Híbrida: Motor Distribuído, Agente Cognitivo e Delta Lake", ha="center", fontsize=10, fontweight="bold", color="#202124")

plt.tight_layout()
plt.savefig("figura1_arquitetura_pipeline.png", dpi=300, bbox_inches="tight")
plt.show()

print("Imagem salva com sucesso como 'figura1_arquitetura_pipeline.png' em 300 DPI!")